# Visualize True vs Predicted Volatility

Notebook này thực hiện hai bước:
1. Gộp toàn bộ file dự đoán trong thư mục `model_results` thành **một file tổng hợp**.
2. Tạo biểu đồ tương tác để so sánh dữ liệu thực và dữ liệu dự đoán theo `dataset`, `model`, `horizon`.

In [1]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Thư mục chứa các file dự đoán riêng lẻ
MODEL_RESULTS_DIR = Path("../model_results")
OUTPUT_PREDICTIONS_CSV = MODEL_RESULTS_DIR / "combined_predictions.csv"

MODEL_RESULTS_DIR.exists(), MODEL_RESULTS_DIR.resolve()

(True,
 WindowsPath('D:/UIT/1003_EPA_PROJECT/1003_EPA-Project_UIT/model_results'))

In [2]:
def normalize_prediction_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Chuan hoa ten cot giua cac file du doan khac schema."""
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    # Bo cot index thua khi CSV luu mac dinh
    unnamed_cols = [c for c in df.columns if c.startswith("Unnamed") or c == ""]
    if unnamed_cols:
        df = df.drop(columns=unnamed_cols)

    rename_map = {}
    for col in df.columns:
        col_lower = col.lower()
        if col_lower == "actual_vol":
            rename_map[col] = "True_Volatility"
        elif col_lower == "pred_vol":
            rename_map[col] = "Pred_Volatility"
        elif col_lower in {"true_volatility", "true_vol"}:
            rename_map[col] = "True_Volatility"
        elif col_lower in {"pred_volatility", "predicted_volatility", "predicted_vol"}:
            rename_map[col] = "Pred_Volatility"
        elif col_lower in {"timestamp", "date", "datetime"}:
            rename_map[col] = "time"

    if rename_map:
        df = df.rename(columns=rename_map)

    required_cols = {
        "dataset",
        "model",
        "horizon",
        "time",
        "True_Volatility",
        "Pred_Volatility",
    }
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Thieu cot {missing}")

    return df


input_csv_paths = sorted(
    p for p in MODEL_RESULTS_DIR.glob("predictions*.csv")
    if p.name != OUTPUT_PREDICTIONS_CSV.name
)

if not input_csv_paths:
    raise FileNotFoundError(f"Khong tim thay file predictions*.csv trong: {MODEL_RESULTS_DIR.resolve()}")

all_frames = []
source_summaries = []

for csv_path in input_csv_paths:
    tmp_df = pd.read_csv(csv_path, low_memory=False)
    tmp_df = normalize_prediction_columns(tmp_df)

    tmp_df["time"] = pd.to_datetime(tmp_df["time"], errors="coerce")
    tmp_df["horizon"] = pd.to_numeric(tmp_df["horizon"], errors="coerce")
    tmp_df = tmp_df.dropna(subset=["time", "horizon"]).copy()
    tmp_df["horizon"] = tmp_df["horizon"].astype(int)

    tmp_df["source_file"] = csv_path.name
    all_frames.append(tmp_df)

    source_summaries.append(
        {
            "source_file": csv_path.name,
            "rows": len(tmp_df),
            "models": int(tmp_df["model"].nunique()),
        }
    )

combined_df = pd.concat(all_frames, ignore_index=True)
combined_df = combined_df.sort_values(["dataset", "model", "horizon", "time"]).reset_index(drop=True)
combined_df.to_csv(OUTPUT_PREDICTIONS_CSV, index=False)

print(f"Da gop {len(input_csv_paths)} file vao: {OUTPUT_PREDICTIONS_CSV}")
print(f"Tong so dong: {len(combined_df):,}")
print("Thong ke theo source:")
for s in source_summaries:
    print(f"- {s['source_file']}: rows={s['rows']:,}, models={s['models']}")

combined_df.head()

Da gop 3 file vao: ..\model_results\combined_predictions.csv
Tong so dong: 427,265
Thong ke theo source:
- predictions_11_4.csv: rows=77,750, models=2
- predictions_fixed_split_data.csv: rows=310,640, models=8
- predictions_moirai_moe_original.csv: rows=38,875, models=1


,dataset,model,horizon,time,True_Volatility,Pred_Volatility,source_file,time_train
0,DAX_40,Autoformer,1,2022-06-06,131.519933,81.265101,predictions_fixed_split_data.csv,20.763631
1,DAX_40,Autoformer,1,2022-06-07,120.885354,91.744866,predictions_fixed_split_data.csv,20.763631
2,DAX_40,Autoformer,1,2022-06-08,122.178167,129.063925,predictions_fixed_split_data.csv,20.763631
3,DAX_40,Autoformer,1,2022-06-09,122.287519,133.226153,predictions_fixed_split_data.csv,20.763631
4,DAX_40,Autoformer,1,2022-06-10,140.701161,165.720726,predictions_fixed_split_data.csv,20.763631


In [3]:
plot_df = pd.read_csv(OUTPUT_PREDICTIONS_CSV)
plot_df["time"] = pd.to_datetime(plot_df["time"], errors="coerce")
plot_df = plot_df.dropna(subset=["time"]).sort_values("time")

dataset_options = sorted(plot_df["dataset"].dropna().unique().tolist())
model_options = sorted(plot_df["model"].dropna().unique().tolist())
horizon_options = sorted(plot_df["horizon"].dropna().unique().astype(int).tolist())

if not dataset_options or not model_options or not horizon_options:
    raise ValueError("No valid data available for interactive visualization")

dataset_selector = widgets.ToggleButtons(
    options=dataset_options,
    description="Dataset:",
)
model_selector = widgets.ToggleButtons(
    options=model_options,
    description="Model:",
)
horizon_selector = widgets.ToggleButtons(
    options=horizon_options,
    description="Horizon:",
    value=1 if 1 in horizon_options else horizon_options[0],
)

def _plot_selected(dataset, model, horizon):
    vis_df = plot_df[
        (plot_df["dataset"] == dataset)
        & (plot_df["model"] == model)
        & (plot_df["horizon"] == int(horizon))
    ].copy()
    vis_df = vis_df.sort_values("time")

    if vis_df.empty:
        print(f"No rows for dataset={dataset}, model={model}, horizon={horizon}")
        return

    plt.figure(figsize=(14, 5))
    plt.plot(
        vis_df["time"],
        vis_df["True_Volatility"],
        label="True Volatility",
        linewidth=2,
    )
    plt.plot(
        vis_df["time"],
        vis_df["Pred_Volatility"],
        label="Predicted Volatility",
        linewidth=2,
    )
    plt.title(f"True vs Predicted Volatility | {dataset} | {model} | horizon={horizon}")
    plt.xlabel("Time")
    plt.ylabel("Volatility")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

ui = widgets.VBox([
    dataset_selector,
    model_selector,
    horizon_selector,
])
out = widgets.interactive_output(
    _plot_selected,
    {
        "dataset": dataset_selector,
        "model": model_selector,
        "horizon": horizon_selector,
    },
)

display(ui, out)

Output()